##Paso 1: Instalación de Dependencias
Instalamos la librería oficial moderna google-genai y la librería requests para la descarga del archivo de datos.

In [1]:
!pip install -q huggingface_hub requests

## Paso 2: Autenticación con Google AI Studio y Secrets de Colab
Para ejecutar llamadas a la API sin exponer la clave en el código:
1. Obtén tu clave gratuita en [Google AI Studio](https://aistudio.google.com/).
2. Abre el panel lateral izquierdo de Colab y selecciona el ícono de llave (**Secrets / Claves**).
3. Crea un secreto con el nombre exacto `GEMINI_API_KEY`, pega tu valor y habilita el interruptor de acceso al cuaderno.

In [5]:
from google.colab import userdata
from google import genai
from google.genai import types
import requests

try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("✅ Cliente de Gemini inicializado correctamente con GEMINI_API_KEY.")
except Exception as e:
    raise RuntimeError(
        "❌ No se encontró 'GEMINI_API_KEY' en Secrets de Colab. "
        "Revisa el panel de la llave a la izquierda y vuelve a intentar."
    )

✅ Cliente de Gemini inicializado correctamente con GEMINI_API_KEY.


##Paso 3: Descarga y Lectura del Documento de Educación

In [18]:
URL_DOCUMENTO = "https://raw.githubusercontent.com/sandragutierrezr/Educacion/refs/heads/main/Reglamento-Estudiantil-CUN.txt"

try:
    response = requests.get(URL_DOCUMENTO, timeout=10)
    if response.status_code == 200:
        documento_educacion = response.text
    else:
        raise ValueError("Respuesta HTTP no exitosa")
except Exception:
    documento_educacion = """REGLAMENTO ESTUDIANTIL DE LA CORPORACIÓN UNIFICADA NACIONAL DE EDUCACIÓN SUPERIOR

1.FORMALIDADES DE INGRESO DEL ESTUDIANTE: Todo aspirante a ingresar a la Corporación debe cumplir con los trámites y requisitos determinados por la
Institución y ceñirse al plan de estudios vigente en los momentos de su ingreso, reintegro o transferencia interna o externa.

2.Valores inscripción: Los valores de inscripción y matrícula y demás derechos pecuniarios de los programas
académicos que oferte la CUN serán fijados, para cada período académico o interperíodo por la Sala General, y no son reembolsables salvo que la Institución decida no abrir el programa al
cual se inscribió el aspirante o medie retiro en los términos dispuestos por este reglamento.

3.RESERVA DEL CUPO: Los aspirantes admitidos por parte de la
Corporación y que no puedan matricularse en el período académico al cual se inscribieron,
podrán solicitar por escrito ante la instancia respectiva la reserva de su cupo para los períodos
siguientes."""

print(f"📄 Documento de educación listo. Total de caracteres: {len(documento_educacion)}")
print("Vista previa inicial:\n", documento_educacion[:350], "...")

📄 Documento de educación listo. Total de caracteres: 107317
Vista previa inicial:
                    ACUERDO No. 002 DE 2011 DEL CONSEJO DIRECTIVO


                             REGLAMENTO ESTUDIANTIL DE LA
           CORPORACIÓN UNIFICADA NACIONAL DE EDUCACIÓN SUPERIOR


                                        LIBRO PRIMERO
                      DEL RÉGIMEN GENERAL DE LOS ESTUDIANTES


                                ...


##Listar Modelos

In [19]:
print("🔍 Listando modelos generativos disponibles en tu cuenta...")
try:
    modelos_disponibles = client.models.list()
    for model in modelos_disponibles:
        print(f"🔹 {model.name}")
except Exception as e:
    print(f"❌ Error al listar modelos: {e}")

🔍 Listando modelos generativos disponibles en tu cuenta...
🔹 models/gemini-2.5-flash
🔹 models/gemini-2.5-pro
🔹 models/gemini-2.5-flash-preview-tts
🔹 models/gemini-2.5-pro-preview-tts
🔹 models/gemma-4-26b-a4b-it
🔹 models/gemma-4-31b-it
🔹 models/gemini-flash-latest
🔹 models/gemini-flash-lite-latest
🔹 models/gemini-pro-latest
🔹 models/gemini-2.5-flash-lite
🔹 models/gemini-2.5-flash-image
🔹 models/gemini-3-flash-preview
🔹 models/gemini-3.1-pro-preview
🔹 models/gemini-3.1-pro-preview-customtools
🔹 models/gemini-3.1-flash-lite-preview
🔹 models/gemini-3.1-flash-lite
🔹 models/gemini-3-pro-image-preview
🔹 models/gemini-3-pro-image
🔹 models/nano-banana-pro-preview
🔹 models/gemini-3.1-flash-image-preview
🔹 models/gemini-3.1-flash-image
🔹 models/gemini-3.1-flash-lite-image
🔹 models/gemini-nano-banana-2.1
🔹 models/gemini-3.5-flash
🔹 models/gemini-3.5-flash-lite
🔹 models/gemini-omni-flash-preview
🔹 models/gemini-omni-1.1-flash
🔹 models/gemini-omni-1.2-flash
🔹 models/gemini-3.5-transcribe
🔹 models/ge

## Paso 4: Configuración de Parámetros de Generación y Sesión de Chat
Definimos el modelo `gemini-2.5-flash` y establecemos la configuración del sistema:
- `system_instruction`: Inyecta el rol analítico y el documento fuente.
- `temperature`: Control de aleatoriedad (0.2 para respuestas deterministas y rigurosas).
- `top_p`: Muestreo por probabilidad acumulada.
- `max_output_tokens`: Límite máximo de longitud de respuesta por turno.

In [20]:
import time

# Lista de modelos gratuitos sugeridos en orden de preferencia
modelos_candidatos = [
    "gemini-3.8-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash-lite"
]

instruccion_sistema = (
    "Eres un asistente educativo analítico especializado para estudiantes de la Cun. "
    "Responde preguntas basándote estricta y prioritariamente en el siguiente documento clínico. "
    "Si el dato solicitado no aparece en el texto, indícalo con claridad antes de dar contexto adicional.\n\n"
    f"--- DOCUMENTO FUENTE ---\n{documento_educacion}\n------------------------"
)

configuracion = types.GenerateContentConfig(
    system_instruction=instruccion_sistema,
    temperature=0.2,
    top_p=0.85,
    max_output_tokens=600
)

chat = None
MODEL_ID = None

print("🤖 Probando disponibilidad de modelos para iniciar el chatbot...")
for modelo in modelos_candidatos:
    try:
        print(f"⏳ Intentando inicializar con {modelo}...")
        chat_test = client.chats.create(
            model=modelo,
            config=configuracion
        )
        # Hacemos una llamada ligera de prueba para validar que no responda con 503 por alta demanda
        chat_test.send_message("Hola")

        # Si la llamada es exitosa, asignamos el chat
        chat = chat_test
        MODEL_ID = modelo
        print(f"✅ ¡Éxito! Sesión de chat iniciada activamente con {MODEL_ID}.")
        break
    except Exception as e:
        print(f"⚠️ El modelo {modelo} no está disponible actualmente o tiene alta demanda. Error: {e}")
        time.sleep(1) # Pequeña pausa antes de intentar con el siguiente candidato

if chat is None:
    raise RuntimeError("❌ No se pudo inicializar ningún modelo disponible en este momento. Inténtalo de nuevo más tarde.")

🤖 Probando disponibilidad de modelos para iniciar el chatbot...
⏳ Intentando inicializar con gemini-3.8-flash...
⚠️ El modelo gemini-3.8-flash no está disponible actualmente o tiene alta demanda. Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 6h43m17.396657773s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/r

## Paso 5: Bucle Interactivo con Auditoría de Tokens en Tiempo Real
Ejecuta esta celda para iniciar la conversación interactiva. El ciclo se mantiene activo hasta que escribas `salir`, `exit` o `terminar`.

In [ ]:
print("=" * 70)
print(f"🤖 CHATBOT EDUCATIVO INTERACTIVO ACTIVO ({MODEL_ID if MODEL_ID else 'Modelo Seleccionado'})")
print("Escribe tu pregunta sobre el documento de salud.")
print("Comandos para terminar: salir | exit | terminar")
print("=" * 70 + "\n")

total_tokens_sesion = 0
turno = 1

while True:
    try:
        pregunta = input("👤 Pregunta: ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\n\nSesión finalizada por el usuario.")
        break

    if not pregunta:
        continue

    if pregunta.lower() in ["salir", "exit", "terminar", "quit"]:
        print(f"\n👋 Sesión cerrada. Consumo acumulado en toda la sesión: {total_tokens_sesion} tokens.")
        break

    try:
        respuesta = chat.send_message(pregunta)

        # Métricas de consumo devueltas en usage_metadata
        tokens_entrada = respuesta.usage_metadata.prompt_token_count
        tokens_salida = respuesta.usage_metadata.candidates_token_count
        tokens_turno = respuesta.usage_metadata.total_token_count
        total_tokens_sesion += tokens_turno

        print(f"\n🩺 Respuesta (Turno {turno}):")
        print(respuesta.text)
        print("-" * 55)
        print(f"📊 Consumo de Tokens:")
        print(f"   • Entrada (Prompt + Historial + Doc): {tokens_entrada}")
        print(f"   • Salida (Tokens generados):          {tokens_salida}")
        print(f"   • Total en este turno:               {tokens_turno}")
        print(f"   • Total acumulado sesión:            {total_tokens_sesion}")
        print("-" * 55 + "\n")

        turno += 1

    except Exception as err:
        print(f"\n⚠️ Error al procesar la solicitud: {err}\n")

🤖 CHATBOT EDUCATIVO INTERACTIVO ACTIVO (gemini-3.5-flash-lite)
Escribe tu pregunta sobre el documento de salud.
Comandos para terminar: salir | exit | terminar

👤 Pregunta: hola

🩺 Respuesta (Turno 1):
¡Hola! ¿En qué puedo ayudarte hoy con información basada en el Reglamento Estudiantil de la CUN?
-------------------------------------------------------
📊 Consumo de Tokens:
   • Entrada (Prompt + Historial + Doc): 27848
   • Salida (Tokens generados):          23
   • Total en este turno:               27871
   • Total acumulado sesión:            27871
-------------------------------------------------------

👤 Pregunta: Cuales son los requisitos de ingreso?

🩺 Respuesta (Turno 2):
Basado en el Artículo 5 del Reglamento Estudiantil (modificado por el Acuerdo No. 002 de 2016 del Consejo Directivo), los requisitos de ingreso para los programas que oferta la Corporación, además de los generales que señale la ley y los específicos de cada programa, son los siguientes:

**1. Para los program